# Autocomplete and Autocorrect Analytics

**Objective:** Build frequency n-gram autocomplete and edit-distance autocorrect; evaluate on a small labelled test set.

> Dataset note: included CSVs are synthetic practice data created for this portfolio. Replace with approved public datasets for official internship submission.


## Setup
Run the following cell. Install missing packages with `pip install -r ../../requirements.txt` from the repository root.

In [ ]:
import re, collections, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
text=Path('text_corpus.txt').read_text(encoding='utf-8').lower()
tokens=re.findall(r'[a-z]+',text)
bigrams=collections.defaultdict(collections.Counter); trigrams=collections.defaultdict(collections.Counter)
for a,b in zip(tokens,tokens[1:]): bigrams[a][b]+=1
for a,b,c in zip(tokens,tokens[1:],tokens[2:]): trigrams[(a,b)][c]+=1
def autocomplete(prefix, top=3):
    key=tuple(prefix.lower().split())
    counts=trigrams.get(key,collections.Counter()) if len(key)==2 else bigrams.get(key[-1],collections.Counter())
    return counts.most_common(top)
tests=['the','data','customers','machine','quick','businesses','products','online','make','learn']
for p in tests: print(p,'->',autocomplete(p))
vocab=set(tokens)
def distance(a,b):
    prev=list(range(len(b)+1))
    for i,ca in enumerate(a,1):
        cur=[i]
        for j,cb in enumerate(b,1): cur.append(min(cur[-1]+1,prev[j]+1,prev[j-1]+(ca!=cb)))
        prev=cur
    return prev[-1]
def autocorrect(word):
    if word in vocab: return word
    return min(vocab,key=lambda w:distance(word,w))
misspellings=['quik','browm','jupms','lzy','bussiness','analysiss','custmers','prodcut','onlne','machne',
              'decisionss','experiance','recieve','teh','datta','helpp','patern','modle','revieew','shoppng']
expected=['quick','brown','jumps','lazy','business','analysis','customers','product','online','machine',
          'decisions','experience','receive','the','data','help','pattern','model','review','shopping']
results=pd.DataFrame({'misspelled':misspellings,'expected':expected,'corrected':[autocorrect(w) for w in misspellings]})
results['correct']=results.corrected==results.expected
display(results); print('Correction accuracy:',results.correct.mean())
freq=collections.Counter(tokens); pd.Series(dict(freq.most_common(20))).plot(kind='bar',title='Top 20 words'); plt.xticks(rotation=45); plt.show()
print('Limitations: tiny corpus, frequency-only context, no personal vocabulary, language/domain dependence. Production keyboards use larger contextual/neural models and user adaptation.')
